# Activity (Advanced, Optional): Estimating Value by Random Rollout
This activity is __advanced and optional__. Value iteration computes the exact optimal value function $V^{\star}$ by sweeping every state and action against the full transition model $P(s^{\prime}\mid s,a)$ and reward $R(s,a)$. When only a simulator of the environment is available, or when an exact full sweep is too costly, an alternative is __rollout__: run a policy forward from a state many times, average the discounted returns, and use the average as an estimate of that state's value. In this activity, you will run random rollouts on the $5\times5$ __slippery__ grid world from the Watch-Demo, quantify how closely a rollout estimate matches the exact $V^{\star}$ from value iteration, and study how the estimate depends on two knobs: the rollout horizon $H$ and the number of simulated trajectories $N$.

> __Learning Objectives.__
>
> In this activity, you will run random rollouts on a slippery grid world, quantify their accuracy against the exact value function, and study how horizon and trajectory count shape the estimate:
> * __Run random rollouts:__ use `rollout_value` to estimate $V^{\pi^{\star}}(s_{0})$ under the optimal policy $\pi^{\star}$ by averaging $N$ simulated trajectory returns.
> * __Quantify rollout accuracy against $V^{\star}$:__ compare the rollout estimate $\hat V^{\pi^{\star}}(s_{0})$ to the exact $V^{\star}(s_{0})$ from value iteration.
> * __Study the effect of horizon $H$ and trajectory count $N$:__ sweep $H$ to see how a truncated horizon biases the estimate, and repeat the estimate across seeds at different $N$ to see how its spread shrinks.

Let's get started.
___

## Theory
For a fixed policy $\pi:\mathcal{S}\rightarrow\mathcal{A}$ acting in a finite MDP $(\mathcal{S},\mathcal{A},P,R,\gamma)$, the state-value function $V^{\pi}:\mathcal{S}\rightarrow\mathbb{R}$ is the expected discounted return starting from $s$ and following $\pi$ thereafter:
$$
V^{\pi}(s) = \mathbb{E}\left[\left.\sum_{t=0}^{\infty}\gamma^{t}R(s_{t},\pi(s_{t}))\;\right|\;s_{0}=s\right],\quad\gamma\in[0,1).
$$
__Rollout__ estimates $V^{\pi}(s_{0})$ from simulated experience instead of an exact sweep. Starting from $s_{0}$, simulate a trajectory of finite horizon $H$ under $\pi$, sampling each next state from $P(\cdot\mid s_{t},\pi(s_{t}))$, and truncate its discounted return at $H$ steps:
$$
G = \sum_{t=0}^{H-1}\gamma^{t}R(s_{t},\pi(s_{t})).
$$
`simulate_return` draws one such trajectory and returns $G$. Repeating this $N$ times independently and averaging gives the __rollout value estimate__:
$$
\hat V^{\pi}(s_{0}) = \frac{1}{N}\sum_{i=1}^{N}G^{(i)},
$$
which `rollout_value` computes directly. By the law of large numbers, $\hat V^{\pi}(s_{0})\rightarrow V^{\pi}(s_{0})$ as $N\rightarrow\infty$. The standard error of the estimate is
$$
\mathrm{SE}\!\left[\hat V^{\pi}(s_{0})\right] = \frac{\sigma_{G}}{\sqrt{N}},
$$
where $\sigma_{G}$ is the standard deviation of a single trajectory return $G$. Accuracy therefore improves as $N$ grows, but only at rate $1/\sqrt{N}$, so quadrupling $N$ only halves the standard error.

Truncating a trajectory at a finite horizon $H$ introduces a second, separate source of error: __bias__. If a trajectory has not reached an absorbing state within $H$ steps, the sum $G=\sum_{t=0}^{H-1}\gamma^{t}R(s_{t},\pi(s_{t}))$ never includes the terminal reward, so it undercounts the return whenever that terminal reward is positive. Increasing $H$ shrinks this bias; increasing $N$ only shrinks the estimator's spread around whatever $H$-dependent quantity it is estimating — it does not fix a horizon that is too short.

The grid world used below is __slippery__: with probability $1-\text{slip}$ the chosen action's move is carried out as intended, and with probability $\text{slip}$ the actual move is a uniformly random one of the four directions instead ($\text{slip}=0.2$ throughout this activity). Because of this randomness, repeated trajectories from the same start state under the same policy visit different cells and earn different returns, so $\hat V^{\pi}(s_{0})$ is itself a random variable that only concentrates around the exact $V^{\pi}(s_{0})$ as $N\rightarrow\infty$; it is not exactly equal to it for any finite $N$.
___

## Setup
This activity uses functions defined in the `src` directory and a small set of external packages. The `include(...)` call below runs `Include.jl`, which activates the local project environment, loads the packages, and includes our code. The first run may take a few minutes while packages are installed and precompiled.

In [1]:
include("Include.jl");

  Activating 

project at `~/Desktop/julia_work/CHEME-140-eCornell-Repository/courses/CHEME-145/module-4`


## Build the Slippery Grid World and the Exact Solution (Reference)
We reuse the $5\times5$ grid world with one goal cell $(5,5)$ (reward $+100$) and one hazard cell $(2,2)$ (reward $-100$); both are absorbing. As in the Watch-Demo, we build the MDP with `slip = 0.2`, so the transition array `mdp.T` is genuinely stochastic: if the intended move or a slip would carry the agent off the grid, the agent instead stays in its current cell and pays only the ordinary step cost. We then solve the resulting MDP exactly with value iteration to obtain $V^{\star}$ and the optimal policy $\pi^{\star}$; these are the ground-truth reference the rollout estimates below are compared against.

__Try changing it:__ after completing the experiments below, come back and change `slip` (e.g., `0.0` for the deterministic grid world, or `0.4` for a noisier one), rebuild `mdp`, and re-run the two experiments to see how the horizon bias and estimator spread respond.

In [2]:
rewards = Dict{Tuple{Int,Int},Float64}((5,5)=>100.0, (2,2)=>-100.0);
world = build(MyRectangularGridWorldModel, (nrows=5, ncols=5, rewards=rewards));
absorbing = Set(keys(rewards));
mdp = build_mdp(world, 0.95; step_reward=-1.0, offgrid_penalty=-1.0, absorbing=absorbing, slip=0.2);
sol = solve(build(MyValueIterationModel, (maxiterations=10_000, ϵ=1e-9)), mdp);
π_star = policy(Q(mdp, sol.V));
sol.V[world.states[(1,1)]]

47.49808762785169

## Experiment 1: Horizon Sensitivity
A rollout truncates each simulated trajectory at $H$ steps (Theory, above). If $H$ is too small for a trajectory to reach the absorbing goal state $(5,5)$ from the start $(1,1)$, the discounted return $G$ never picks up the $+100$ terminal reward, so $\hat V^{\pi^{\star}}(s_{0})$ undercounts the true value and comes in __biased low__ relative to the exact $V^{\star}(s_{0})$. The cell below fixes $N=2{,}000$ and the random seed, and sweeps $H\in\{10,25,50,100,200\}$. For small $H$ the horizon bias dominates and the estimate rises sharply as $H$ grows; once $H$ is large enough for a trajectory to reach the goal, that bias is gone and the small run-to-run wobble is ordinary Monte Carlo noise (Experiment 2), not a horizon effect. Some large-$H$ estimates even land slightly above $V^{\star}(s_{0})$, which horizon truncation alone could never produce.

__Try changing it:__ add a larger value of `H` (e.g., `400`) to the tuple below, or lower `N`, and re-run to see how the estimate moves.

In [3]:
let
    s0 = world.states[(1,1)];
    for H ∈ (10, 25, 50, 100, 200)
        est = rollout_value(mdp, s0; π_fn = s -> π_star[s], H = H, N = 2_000, rng = Random.MersenneTwister(3));
        println("H = $(H):  estimate = ", round(est, digits = 3), "   (exact ", round(sol.V[s0], digits = 3), ")");
    end
end

H = 10:  estimate = 35.548

   (exact 47.498)
H = 25:  estimate = 48.687

   (exact 47.498)
H = 50:  estimate = 48.816   (exact 47.498)
H = 100:  estimate = 47.138

   (exact 47.498)
H = 200:  estimate = 46.532

   (exact 47.498)


## Experiment 2: Estimator Variance
For a fixed $H$, repeating the rollout estimate with different random seeds samples the estimator $\hat V^{\pi^{\star}}(s_{0})$ itself; the spread of these repeats approximates its standard error $\mathrm{SE}[\hat V^{\pi^{\star}}(s_{0})]=\sigma_{G}/\sqrt{N}$ (Theory, above). The cell below repeats the estimate at $H=200$ across ten seeds, first at $N=200$ and then at $N=1{,}000$, and reports the mean and standard deviation of the ten estimates at each $N$; the standard deviation should be smaller at the larger $N$.

__Try changing it:__ add a third, larger `N` (e.g., `5_000`) to the tuple below, or increase the number of seeds, and re-run to see the spread shrink further.

In [4]:
let
    s0 = world.states[(1,1)];
    for N ∈ (200, 1_000)
        ests = [ rollout_value(mdp, s0; π_fn = s -> π_star[s], H = 200, N = N, rng = Random.MersenneTwister(seed)) for seed ∈ 1:10 ];
        println("N = $(N):  mean = ", round(mean(ests), digits = 3), "   std = ", round(std(ests), digits = 3));
    end
end

N = 200:  mean = 46.998

   std = 2.009
N = 1000:  mean = 

47.277   std = 0.863


## Summary
This activity ran random rollouts on the $5\times5$ slippery grid world ($\text{slip}=0.2$), swept the rollout horizon $H$ to see how a truncated horizon biases the value estimate, and repeated the estimate across seeds at two values of $N$ to see how its spread shrinks.

> __Key Takeaways:__
>
> * **Too-short horizons bias the estimate low:** when $H$ is too small for a trajectory to reach the absorbing goal within $H$ steps, the truncated return $G=\sum_{t=0}^{H-1}\gamma^{t}R(s_{t},\pi(s_{t}))$ never picks up the $+100$ terminal reward, so $\hat V^{\pi^{\star}}(s_{0})$ comes in below the exact $V^{\star}(s_{0})$; the bias shrinks as $H$ grows.
> * **Estimator variance shrinks with $N$, but only at rate $1/\sqrt{N}$:** repeating the rollout estimate across seeds at $H=200$ gives a smaller standard deviation at $N=1{,}000$ than at $N=200$, consistent with $\mathrm{SE}[\hat V^{\pi^{\star}}(s_{0})]=\sigma_{G}/\sqrt{N}$.
> * **Rollout trades exactness for a model-light estimate:** unlike value iteration, which sweeps the full transition model $P(s^{\prime}\mid s,a)$, `rollout_value` only needs a simulator and a policy, at the cost of horizon bias and Monte Carlo noise.

Rollout estimates a state's value from simulated trajectories rather than an exact sweep over the full transition model, trading the exactness of value iteration for an estimate whose accuracy depends on both the rollout horizon $H$ and the number of trajectories $N$.
___

### Additional Resources
* Sutton, R. S., & Barto, A. G. (2018). _Reinforcement Learning: An Introduction_ (2nd ed.), Chapter 8. MIT Press.
* Kochenderfer, M. J., Wheeler, T. A., & Wray, K. H. (2022). _Algorithms for Decision Making_. MIT Press.
* Bertsekas, D. P. (2020). _Rollout, Policy Iteration, and Distributed Reinforcement Learning_. Athena Scientific.